In [1]:
import pandas as pd
train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")
print(train.shape, test.shape)
print("pehle 3 columns:", train.columns[:3].tolist())
print("aakhri 5 columns:", train.columns[-5:].tolist())

(2165, 66) (562, 66)
pehle 3 columns: ['label', 'signer_id', 'filename']
aakhri 5 columns: ['f58', 'f59', 'f60', 'f61', 'f62']


In [2]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from src.features import add_engineered_features

A, B = "Tuey", "Daal"

train = pd.read_csv("../data/processed/train.csv")
FEATS = [f"f{i}" for i in range(63)]
X = train[FEATS].values
y = train["label"].values
g = train["signer_id"].values

assert A in set(y) and B in set(y), f"Label ka naam alag hai! Milte julte: {[l for l in set(y) if l.lower()[:2] in ('tu', 'da')]}"
print(train["label"].value_counts()[[A, B]])
print("Signers jinke paas Tuey hai:", train[train.label == A].signer_id.nunique(),
      "| Daal:", train[train.label == B].signer_id.nunique(),
      "| dono:", len(set(train[train.label == A].signer_id) & set(train[train.label == B].signer_id)))

label
Tuey    60
Daal    60
Name: count, dtype: int64
Signers jinke paas Tuey hai: 60 | Daal: 60 | dono: 56


In [3]:
def mirror(X):
    M = X.copy()
    M[:, 0::3] *= -1            # left haath ke liye: har x ulta
    return M

def make_v3():
    return Pipeline([("eng", FunctionTransformer(add_engineered_features)),
                     ("sc", StandardScaler()),
                     ("svm", SVC(C=100, gamma=0.01))])

oof = np.empty(len(y), dtype=object)
for tr, va in GroupKFold(n_splits=5).split(X, y, g):
    Xtr = np.vstack([X[tr], mirror(X[tr])])
    ytr = np.concatenate([y[tr], y[tr]])
    oof[va] = make_v3().fit(Xtr, ytr).predict(X[va])

print(f"36-class model, signer-wise CV accuracy: {(oof == y).mean():.4f}\n")
for L, other in ((A, B), (B, A)):
    m = y == L
    print(f"{L}: {m.sum()} images, recall = {(oof[m] == L).mean():.2f}")
    print(f"   -> {other} bola gaya: {(oof[m] == other).mean():.2f}   |  koi aur harf bola gaya: {((oof[m] != L) & (oof[m] != other)).mean():.2f}")
    print("   sab se aam ghalat jawab:", pd.Series(oof[m][oof[m] != L]).value_counts().head(4).to_dict())

36-class model, signer-wise CV accuracy: 0.9524

Tuey: 60 images, recall = 0.58
   -> Daal bola gaya: 0.42   |  koi aur harf bola gaya: 0.00
   sab se aam ghalat jawab: {'Daal': 25}
Daal: 60 images, recall = 0.65
   -> Tuey bola gaya: 0.35   |  koi aur harf bola gaya: 0.00
   sab se aam ghalat jawab: {'Tuey': 21}


In [4]:
pair = np.isin(y, [A, B])
Xp, yp, gp = X[pair], y[pair], g[pair]
base_acc = (oof[pair] == yp).mean()
print(f"Baseline (36-class model) in {pair.sum()} images par: {base_acc:.3f}")
print(f"Chance (hamesha bara class bolo): {max((yp == A).mean(), (yp == B).mean()):.3f}\n")

def specialist(kind):
    if kind == "SVM":
        clf = SVC(C=100, gamma=0.01, class_weight="balanced")
    elif kind == "LogReg":
        clf = LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced")
    else:
        clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
    return Pipeline([("eng", FunctionTransformer(add_engineered_features)), ("sc", StandardScaler()), ("clf", clf)])

n_splits = min(5, len(set(gp)))
rows = []
for kind in ("SVM", "LogReg", "RF"):
    pred = np.empty(len(yp), dtype=object)
    for tr, va in GroupKFold(n_splits=n_splits).split(Xp, yp, gp):
        Xtr = np.vstack([Xp[tr], mirror(Xp[tr])])
        ytr = np.concatenate([yp[tr], yp[tr]])
        pred[va] = specialist(kind).fit(Xtr, ytr).predict(Xp[va])
    rows.append({"model": kind, "accuracy": (pred == yp).mean(),
                 f"{A} recall": (pred[yp == A] == A).mean(), f"{B} recall": (pred[yp == B] == B).mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nSamajh: agar specialist ka accuracy baseline se kaafi upar nahi, to landmarks se farq nikalna mushkil hai.")

Baseline (36-class model) in 120 images par: 0.617
Chance (hamesha bara class bolo): 0.500

 model  accuracy  Tuey recall  Daal recall
   SVM     0.542        0.550        0.533
LogReg     0.517        0.450        0.583
    RF     0.492        0.483        0.500

Samajh: agar specialist ka accuracy baseline se kaafi upar nahi, to landmarks se farq nikalna mushkil hai.


In [5]:
Xe = add_engineered_features(Xp)                   # (n, 93)

def fname(i):
    if i < 63:
        return f"lm{i // 3}_{'xyz'[i % 3]}"
    j = i - 63
    if j < 5:
        return f"wrist_tip#{j}"
    if j < 15:
        return f"tip_tip#{j - 5}"
    return f"angle#{j - 15}"

a, b = Xe[yp == A], Xe[yp == B]
d = (b.mean(0) - a.mean(0)) / np.sqrt((a.var(0) + b.var(0)) / 2 + 1e-12)       # Cohen's d (Daal - Tuey)

# same signer ke andar farq: kya har signer mein farq ek hi taraf jata hai?
df = pd.DataFrame(Xe)
df["y"], df["g"] = yp, gp
both = [s for s in set(gp) if {A, B} <= set(df[df.g == s].y)]
diff = np.array([df[(df.g == s) & (df.y == B)].drop(columns=["y", "g"]).mean().values -
                 df[(df.g == s) & (df.y == A)].drop(columns=["y", "g"]).mean().values for s in both])
same_sign = np.maximum((diff > 0).mean(0), (diff < 0).mean(0))

top = np.argsort(-np.abs(d))[:10]
out = pd.DataFrame({"feature": [fname(i) for i in top], "farq (d)": d[top].round(2),
                    "signers jin mein farq ek hi taraf": (same_sign[top] * 100).round(0).astype(int).astype(str) + "%"})
print(f"{len(both)} signers ne dono harf kiye.  |d| 0.2 = chhota, 0.5 = darmiyana, 0.8 = bara farq\n")
print(out.to_string(index=False))

56 signers ne dono harf kiye.  |d| 0.2 = chhota, 0.5 = darmiyana, 0.8 = bara farq

  feature  farq (d) signers jin mein farq ek hi taraf
tip_tip#9      0.40                               64%
   lm10_y      0.32                               57%
 angle#11      0.31                               66%
tip_tip#2      0.31                               61%
   lm14_y      0.30                               64%
    lm8_y      0.29                               61%
   lm13_y      0.28                               61%
tip_tip#8      0.28                               62%
    lm4_z      0.28                               66%
    lm7_y      0.28                               55%


## Nateeja
Tuey aur Daal MediaPipe landmarks se alag nahi hote: 36-class CV accuracy 0.9524, lekin in dono par sirf 0.617 (chance 0.5, n=120, ±4.5 pt). Alag specialist (SVM 0.542, LogReg 0.517, RF 0.492) koi behtari nahi laya. Top feature ka d = 0.40, aur 56 signers mein se sirf 55-66% mein farq ek hi taraf. Kul ghaltiyon ka ~45% yehi jori hai; dono ko ek group maanein to accuracy ~97.4%. Live app mein is jori ko "ambiguous group" maan kar user se confirm karwate hain.

In [6]:
from collections import Counter

# signer-wise CV mein sab se zyada ghalat jodiyan (test.csv ko haath nahi lagaya)
wrong = [(t, p) for t, p in zip(y, oof) if t != p]
pairs = Counter(tuple(sorted(tp)) for tp in wrong)
print(f"Kul ghaltiyan: {len(wrong)} / {len(y)}\n")
print("Sab se zyada confuse hone wali jodiyan (donon taraf ki ghaltiyan jod kar):")
for (a, b), n in pairs.most_common(12):
    ab = sum(1 for t, p in wrong if (t, p) == (a, b))
    ba = sum(1 for t, p in wrong if (t, p) == (b, a))
    print(f"  {a:>8} <-> {b:<8}  {n:3d}  ({a}->{b}: {ab}, {b}->{a}: {ba})")

# sab se kamzor akele harf
rec = pd.Series({L: (oof[y == L] == L).mean() for L in sorted(set(y))}).sort_values()
print("\nSab se kam recall wale 10 harf:")
print(rec.head(10).round(2).to_string())

Kul ghaltiyan: 103 / 2165

Sab se zyada confuse hone wali jodiyan (donon taraf ki ghaltiyan jod kar):
      Daal <-> Tuey       46  (Daal->Tuey: 21, Tuey->Daal: 25)
       Ain <-> Ghain       8  (Ain->Ghain: 5, Ghain->Ain: 3)
      Zuad <-> Zuey        7  (Zuad->Zuey: 5, Zuey->Zuad: 2)
      Seen <-> Taay        5  (Seen->Taay: 3, Taay->Seen: 2)
      Alif <-> Suad        4  (Alif->Suad: 3, Suad->Alif: 1)
      Meem <-> Taay        3  (Meem->Taay: 2, Taay->Meem: 1)
  Dochahay <-> Wao         2  (Dochahay->Wao: 2, Wao->Dochahay: 0)
  Dochahay <-> Sheen       2  (Dochahay->Sheen: 1, Sheen->Dochahay: 1)
       Kaf <-> Pay         2  (Kaf->Pay: 1, Pay->Kaf: 1)
      Meem <-> Nuun        2  (Meem->Nuun: 1, Nuun->Meem: 1)
      Alif <-> Taay        1  (Alif->Taay: 1, Taay->Alif: 0)
       Bay <-> Ray         1  (Bay->Ray: 1, Ray->Bay: 0)

Sab se kam recall wale 10 harf:
Tuey        0.58
Daal        0.65
Ain         0.90
Ghain       0.91
Taay        0.92
Zuad        0.92
Dochahay    0.93
Seen

In [7]:
import os
os.makedirs("../results", exist_ok=True)

rows = []
for (a, b), n in pairs.most_common(20):
    ab = sum(1 for t, p in wrong if (t, p) == (a, b))
    ba = sum(1 for t, p in wrong if (t, p) == (b, a))
    rows.append({"harf_a": a, "harf_b": b, "ghaltiyan": n, "a_to_b": ab, "b_to_a": ba})
pd.DataFrame(rows).to_csv("../results/advanced_confusion_pairs_cv.csv", index=False)
rec.round(3).rename("recall").to_csv("../results/advanced_per_letter_recall_cv.csv")
print("save ho gaya:", len(rows), "jodiyan")

save ho gaya: 20 jodiyan
